# Phase 5: Recommendation Engine & Diagnostic Evaluation

**Project:** Indian E-Commerce Recommendation System  
**Objective:** Test the Content-Based Recommendation Engine using sparse TF-IDF Cosine Similarity, verify operational guarantees, and benchmark real-time latency.

### Diagnostic Standards Verified:
1. **Strict Self-Exclusion**: A viewed product must never recommend itself ($0.0\%$ self-recommendation rate).
2. **Score Bounds**: Cosine Similarity scores strictly bounded in $[0.0, 1.0]$.
3. **Zero Duplicate Recommendations**: All Top-N items returned must be distinct products.
4. **Intra-Category Cohesion**: High semantic alignment within retail departments.
5. **Real-time Query Latency**: Sub-10ms response time per recommendation query.

In [ ]:
import os
import sys
import json
import time
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))
from src.models.recommender import get_default_recommender

recommender = get_default_recommender()
print(f"Recommender online: {len(recommender.catalog_df):,} products indexed.")

## 1. Load Evaluation Diagnostics Report

In [ ]:
EVAL_JSON = os.path.join("..", "reports", "recommendation_evaluation.json")
with open(EVAL_JSON, "r", encoding="utf-8") as f:
    report = json.load(f)

print(f"Evaluated Test Sample:             {report['evaluation_sample_size']} items")
print(f"Self-Recommendation Rate:          {report['self_recommendation_rate_pct']}% (Target: 0.0%)")
print(f"Duplicate Recommendation Rate:     {report['duplicate_recommendation_rate_pct']}% (Target: 0.0%)")
print(f"Intra-Category Cohesion:           {report['intra_category_cohesion_pct']}%")
print(f"Mean Query Latency:                {report['latency_ms']['mean']} ms")
print(f"P95 Query Latency:                 {report['latency_ms']['p95']} ms")

## 2. Interactive Product Recommendation Demo
Demonstrating recommendations for a specific product ID.

In [ ]:
# Example: Querying the first laptop in catalog
laptop_sample = recommender.catalog_df[recommender.catalog_df["category"] == "Laptops & Computers"].iloc[0]
sample_id = laptop_sample["product_id"]

print(f"Query Product: {laptop_sample['title']} (INR {laptop_sample['price_inr']:,})")
print("=" * 80)

recs = recommender.recommend_by_product_id(sample_id, top_n=5)
for r in recs:
    print(f"Rank #{r['rank']} | Sim: {r['similarity_score']:.4f} | ₹{r['price_inr']:,}")
    print(f"   [{r['category']}] {r['title']}")
    print("-" * 80)

## 3. Free-Text Semantic Search & Recommendation
Vectorizes arbitrary user search queries and matches against catalog vectors.

In [ ]:
query = "wireless bluetooth noise cancelling headphones"
text_recs = recommender.recommend_by_text(query, top_n=5)
print(f"Search Query: '{query}'")
print("=" * 80)
for r in text_recs:
    print(f"Rank #{r['rank']} | Sim: {r['similarity_score']:.4f} | ₹{r['price_inr']:,}")
    print(f"   [{r['category']}] {r['title']}")
    print("-" * 80)